# Simple: Group Variability Analysis with PAMOLA.CORE

**Goal**: Learn group variance analysis basics in 5 minutes using operation.execute()

**What you'll learn:**
- Load sample data from examples/data_examples/
- Analyze variability within grouped records using execute()
- Understand variance metrics for anonymization decisions
- Identify groups suitable for aggregation

**Prerequisites:**
- Python 3.8+
- PAMOLA.CORE installed
- Basic pandas knowledge

---

## Step 1: Setup and Imports

**How to use:**
- Run this cell to import required libraries
- Auto-detects PAMOLA installation path (searches up 6 levels)
- If not found locally, auto-installs from GitHub

**What you'll see:**
- Detection message showing PAMOLA location or installation progress
- Import confirmation (✅ All imports successful!)
- Notebook start timestamp
- Project root and working directory paths

**Note:** First run may take 1-2 minutes if installing from GitHub

**Expected structure:**
```
PAMOLA/
├── pamola_core/          ← Auto-detected
└── examples/
    ├── data_examples/    ← Data directory
    └── profiling/analyzers/
        └── 01_group_analyzer_simple.ipynb  ← You are here
```

In [ ]:
import pandas as pd
import numpy as np
import sys
import os
import json
from pathlib import Path
from datetime import datetime
from IPython.display import Image, display
 
print("🔍 Detecting PAMOLA installation...\n") 
 
# Auto-detect project root (search up 6 levels for pamola_core/) 
current_dir = Path.cwd() 
project_root = current_dir 
pamola_found = False

for level in range(6): 
    if (project_root / 'pamola_core').exists(): 
        print(f"✓ Found pamola_core at level {level}: {project_root}") 
        sys.path.insert(0, str(project_root))
        pamola_found = True
        break 
    project_root = project_root.parent 

# If not found locally, try to install from Git
if not pamola_found:
    print("⚠️  pamola_core not found in project structure")
    print("📦 Attempting to install from GitHub...\n")
    
    try:
        import subprocess
        
        # Install from Git repository
        install_cmd = [
            sys.executable, "-m", "pip", "install",
            "git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3"
        ]
        
        result = subprocess.run(
            install_cmd,
            capture_output=True,
            text=True,
            check=True
        )
        
        print("✅ Successfully installed pamola_core from GitHub!")
        print(f"   Revision: pre-epic3")
        
    except subprocess.CalledProcessError as e:
        print(f"❌ Installation failed!")
        print(f"   Error: {e.stderr}")
        raise RuntimeError(
            "Could not find pamola_core locally and installation from GitHub failed. "
            "Please install manually using:\n"
            "pip install git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3"
        )
    except Exception as e:
        print(f"❌ Unexpected error during installation: {e}")
        raise

# Now try to import the required modules
try:
    from pamola_core.profiling.analyzers.group import GroupAnalyzerOperation
    from pamola_core.utils.ops.op_data_source import DataSource
    from pamola_core.utils.progress import HierarchicalProgressTracker
    from pamola_core.utils.tasks.task_reporting import TaskReporter
    from pamola_core.io.csv import read_csv
    
    print("✅ All imports successful!")
    print(f"📅 Notebook started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("=" * 80)
    print(f"📂 Project root:  {project_root.name}")
    print(f"📂 Working dir:   {current_dir.relative_to(project_root)}")
    print("=" * 80)
    
except ImportError as e:
    print(f"❌ Import failed: {e}")
    print("\n💡 Troubleshooting:")
    print("   1. Try restarting your Python kernel/notebook")
    print("   2. Verify installation: pip list | grep pamola")
    print("   3. Manual install: pip install git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3")
    raise

## Step 2: Load Sample Data

**How to use:**
- Run to load data from examples/data_examples/sample.csv
- Auto-creates sample data if file doesn't exist
- Review dataset structure and statistics

**What you'll see:**
- File path confirmation or creation message
- Dataset summary (30 records, 5 columns)
- First 10 records preview
- Column statistics (types, unique values, ranges)
- Grouped data: multiple resumes per person_id (1-4 resumes per person)

**Note:** Sample includes grouped records with varying attributes to demonstrate variance analysis

In [ ]:
examples_dir = project_root / 'examples'
data_path = examples_dir / 'data_examples' / 'sample.csv'

if not data_path.exists():
    print("⚠️  File not found, creating sample data...")
    data_path.parent.mkdir(parents=True, exist_ok=True)
    
    # Create sample data with person IDs and varying attributes
    sample_data = pd.DataFrame({
        'resume_id': range(1, 31),
        'person_id': [1, 1, 1, 2, 2, 3, 3, 3, 4, 4,
                     5, 5, 6, 6, 6, 7, 8, 8, 9, 9,
                     10, 10, 10, 11, 11, 12, 13, 13, 14, 15],
        'name': ['Alice Smith', 'Alice Smith', 'Alice S.',
                'Bob Jones', 'Bob Jones',
                'Charlie Brown', 'Charlie Brown', 'C. Brown',
                'Diana Prince', 'Diana Prince',
                'Eve Adams', 'Eve A.',
                'Frank Miller', 'Frank Miller', 'Frank M.',
                'Grace Lee',
                'Henry Wilson', 'Henry Wilson',
                'Iris Chen', 'Iris Chen',
                'Jack Taylor', 'Jack Taylor', 'J. Taylor',
                'Karen White', 'Karen White',
                'Leo Martin',
                'Mia Davis', 'Mia Davis',
                'Nathan Garcia',
                'Olivia Martinez'],
        'title': ['Software Engineer', 'Software Engineer', 'Senior Software Engineer',
                 'Data Scientist', 'Data Scientist',
                 'Product Manager', 'Product Manager', 'Product Manager',
                 'UX Designer', 'UX Designer',
                 'Marketing Manager', 'Marketing Manager',
                 'Sales Executive', 'Sales Executive', 'Sales Executive',
                 'HR Manager',
                 'Financial Analyst', 'Financial Analyst',
                 'DevOps Engineer', 'DevOps Engineer',
                 'QA Engineer', 'QA Engineer', 'Senior QA Engineer',
                 'Content Writer', 'Content Writer',
                 'Business Analyst',
                 'Project Manager', 'Project Manager',
                 'IT Support',
                 'Operations Manager'],
        'experience_years': [3, 3, 5, 4, 4, 6, 6, 7, 2, 2,
                           8, 8, 5, 5, 5, 10, 3, 3, 4, 4,
                           6, 6, 8, 4, 4, 7, 5, 5, 2, 9],
        'location': ['San Francisco', 'San Francisco', 'San Francisco',
                    'New York', 'New York',
                    'Chicago', 'Chicago', 'Chicago',
                    'Seattle', 'Seattle',
                    'Boston', 'Boston',
                    'Austin', 'Austin', 'Austin',
                    'Denver',
                    'Portland', 'Portland',
                    'Los Angeles', 'Los Angeles',
                    'Miami', 'Miami', 'Miami',
                    'Phoenix', 'Phoenix',
                    'Dallas',
                    'Atlanta', 'Atlanta',
                    'Philadelphia',
                    'San Diego']
    })
    sample_data.to_csv(data_path, index=False)
    print(f"✓ Sample data created")

df = read_csv(data_path)
print(f"📊 Dataset loaded: {len(df)} records, {len(df.columns)} columns")
print(f"\n🔍 First 10 records:")
print(df.head(10))

print(f"\n📈 Column Statistics:")
for col in df.columns:
    dtype_str = str(df[col].dtype)
    if pd.api.types.is_string_dtype(df[col]):
        print(f"  {col:20s} ({dtype_str:10s}): {df[col].nunique()} unique values")
    else:
        print(f"  {col:20s} ({dtype_str:10s}): min={df[col].min()}, max={df[col].max()}")

## Step 3: Setup Task Environment

**How to use:**
1. **CUSTOMIZE field_name and fields_config**: Edit `create_config_kwargs()` function
   - Change `"field_name": "resume_id"` to YOUR grouping column (or keep default)
   - Change `"fields_config"` to YOUR analysis fields with weights
   - Available fields: person_id, name, title, experience_years, location
2. Run to validate fields and setup environment

**What you'll see:**
- Grouping field validation (✓ Field exists, unique groups, records per group)
- Analysis fields validation (✓ All fields exist with unique value counts)
- Task directory created (✓)
- TaskReporter initialized (✓)
- Config kwargs confirmed (✓)
- DataSource created (✓)
- Progress tracker ready (✓)

**Note:** Default fields don't exist - you MUST customize both field_name and fields_config to match sample data columns!

**Configuration pattern (production-style):**
```python
def create_config_kwargs():
    return {
        "dataset_name": "main_dataset",
        "field_name": "person_id",  # Group by this field
        "fields_config": {           # Fields to analyze with weights
            "name": 2,               # Higher weight = more important
            "title": 1,
            "location": 1
        }
    }
```

In [ ]:
# Configuration helper function (production pattern)
def create_config_kwargs():
    return {
        "dataset_name": "main_dataset",
        "field_name": "resume_id",  # Group by this field - CUSTOMIZE THIS!
        "fields_config": {            # Fields to analyze with weights - CUSTOMIZE THIS!
            "first_name": 2,
            "job_title_current": 1,
            "location_city": 1
        }
    }

kwargs = create_config_kwargs()
field_name = kwargs["field_name"]
fields_config = kwargs["fields_config"]

# Validate field exists
print(f"\n🔍 Validating field configuration...\n")
if field_name not in df.columns:
    raise ValueError(
        f"❌ Field '{field_name}' not found in dataset!\n"
        f"Available columns: {', '.join(df.columns)}\n"
        f"Please update 'field_name' in create_config_kwargs()"
    )

# Validate analysis fields exist
missing_fields = [f for f in fields_config.keys() if f not in df.columns]
if missing_fields:
    raise ValueError(
        f"❌ Analysis fields not found: {missing_fields}\n"
        f"Available columns: {', '.join(df.columns)}\n"
        f"Please update 'fields_config' in create_config_kwargs()"
    )

print(f"✓ Grouping field: '{field_name}'")
print(f"  Unique groups: {df[field_name].nunique()}")
print(f"  Records per group: {len(df) / df[field_name].nunique():.2f} avg")

print(f"\n✓ Fields to analyze (with weights):")
for field, weight in fields_config.items():
    print(f"  {field:20s}: weight={weight}")

# Create task directory
task_dir = examples_dir / 'data_examples' / 'simple_output'
os.makedirs(task_dir, exist_ok=True)
print(f"\n✓ Task directory: {task_dir}")

# Initialize TaskReporter
task_reporter = TaskReporter(
    task_id="simple_001",
    task_type="group_analysis",
    description=f"Group variability analysis for '{field_name}'",
    report_path=task_dir
)
print("✓ TaskReporter initialized")

execute_kwargs = {"dataset_name": kwargs["dataset_name"]}
print(f"✓ Config kwargs: {execute_kwargs}")

# Create DataSource
data_source = DataSource(dataframes={"main_dataset": df})
print("✓ DataSource created")

# Setup progress tracker
tracker = HierarchicalProgressTracker(
    total=6,
    description=f"Analyzing {field_name} groups",
    unit="steps",
    track_memory=True,
    level=0,
    bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}]"
)
print("✓ Progress tracker ready")

print(f"\n✅ Environment setup complete!")

## Step 4: Configure & Execute Operation

**How to use:**
- Review operation configuration
- Run to execute group variability analysis
- Monitor progress bar (6 processing steps)

**What you'll see:**
- Configuration summary (grouping field, analysis fields, thresholds)
- Progress bar: validate → group → calculate variance → assess → visualize → save
- Completion message with status and artifact count (2-4 files expected)
- Status = "success" (verify no errors)

**Key parameters:**
- `field_name`: Field to group records by (from config)
- `fields_config`: Fields to analyze with importance weights
- `variance_threshold=0.2`: Max variance for aggregation eligibility
- `hash_algorithm='md5'`: Text comparison method
- `generate_visualization=True`: Create variance distribution charts

In [ ]:
# Create operation with production-style configuration
operation = GroupAnalyzerOperation(
    # Core parameters
    field_name=field_name,              # Field to group by
    fields_config=fields_config,        # Fields to analyze with weights
    
    # Variance thresholds
    variance_threshold=0.2,             # Max variance for aggregation
    large_group_threshold=100,          # Size threshold for large groups
    large_group_variance_threshold=0.05,# Stricter threshold for large groups
    
    # Text processing
    text_length_threshold=100,          # Min length for hashing
    hash_algorithm='md5',               # md5 or minhash
    minhash_similarity_threshold=0.7,   # Similarity threshold for minhash
    
    # Processing settings
    use_cache=False,
    
    # Execution behavior
    generate_visualization=True,        # Create visualization artifacts
    save_output=True,                   # Save results to files
    force_recalculation=False           # Use cache when available
)

print("✓ Operation configured")
print(f"  Grouping field:           {operation.field_name}")
print(f"  Fields to analyze:        {len(operation.fields_config)}")
print(f"  Variance threshold:       {operation.variance_threshold}")
print(f"  Large group threshold:    {operation.large_group_threshold}")
print(f"  Hash algorithm:           {operation.hash_algorithm}")
print(f"  Visualizations:           {operation.generate_visualization}")
print(f"  Save output:              {operation.save_output}")
print(f"  Force recalc:             {operation.force_recalculation}")

# Execute operation
print("\n" + "=" * 80)
print("⚙️  Executing group variability analysis...")
print("=" * 80 + "\n")

result = operation.execute(
    data_source=data_source,
    task_dir=task_dir,
    reporter=task_reporter,
    progress_tracker=tracker,
    **execute_kwargs
)

print("\n" + "=" * 80)
print(f"✅ Operation completed!")
print(f"   Status: {result.status}")
print(f"   Artifacts: {len(result.artifacts)}")
print("=" * 80)

## Step 5: Analyze Results

**How to use:**
- Run to load and display group variance results
- Review variance statistics and aggregation recommendations
- Check field-specific contributions to variance

**What you'll see:**
- Generated artifacts list (metrics JSON, visualizations)
- Overall statistics (total groups, average/max variance, aggregation potential)
- Variance distribution (breakdown by threshold ranges)
- Field-specific metrics (per-field variance and duplication ratios)
- Algorithm configuration details
- Summary metrics from operation result

**Note:** Groups with variance ≤ threshold can be safely aggregated. Lower variance = more similar records within group.

**Generated artifacts:**
- Metrics JSON in metrics/
- Visualizations in visualizations/

In [ ]:
# Display generated artifacts
if result.artifacts:
    print("\n📦 Generated Artifacts:")
    print("=" * 80)
    for artifact in result.artifacts:
        print(f"   [{artifact.category}] {artifact.artifact_type}: {artifact.path.name}")

# Find and load metrics file
metrics_files = list(task_dir.glob('metrics/*_metrics.json'))
if metrics_files:
    metrics_file = metrics_files[0]
    
    with open(metrics_file, 'r') as f:
        metrics = json.load(f)
    
    print("\n" + "=" * 80)
    print("📊 GROUP VARIABILITY ANALYSIS RESULTS")
    print("=" * 80)
    
    # Overall statistics
    print("\n📈 Overall Statistics:")
    print(f"  Total groups analyzed:  {metrics.get('total_groups', 0):,}")
    print(f"  Total records:          {metrics.get('total_records', 0):,}")
    print(f"  Average variance:       {metrics.get('avg_variance', 0):.4f}")
    print(f"  Maximum variance:       {metrics.get('max_variance', 0):.4f}")
    print(f"  Groups to aggregate:    {metrics.get('groups_to_aggregate', 0):,}")
    
    # Aggregation potential
    if metrics.get('total_groups', 0) > 0:
        agg_pct = (metrics.get('groups_to_aggregate', 0) / metrics.get('total_groups', 1)) * 100
        print(f"  Aggregation potential:  {agg_pct:.1f}%")
    
    # Variance distribution
    if 'threshold_metrics' in metrics:
        print("\n📊 Variance Distribution:")
        dist = metrics['threshold_metrics']
        labels = {
            'below_0.1': '< 0.1 (Very Low)',
            '0.1_to_0.2': '0.1 - 0.2 (Low)',
            '0.2_to_0.5': '0.2 - 0.5 (Medium)',
            '0.5_to_0.8': '0.5 - 0.8 (High)',
            'above_0.8': '> 0.8 (Very High)'
        }
        for key, label in labels.items():
            count = dist.get(key, 0)
            pct = (count / metrics.get('total_groups', 1)) * 100
            bar = '█' * int(pct / 2)
            print(f"  {label:25s}: {bar:20s} {count:3d} ({pct:5.1f}%)")
    
    # Field-specific metrics
    if 'field_metrics' in metrics:
        print("\n📋 Field-Specific Metrics:")
        for field, field_stats in metrics['field_metrics'].items():
            print(f"\n  {field.upper()}:")
            print(f"    Avg variance:           {field_stats.get('avg_variance', 0):.4f}")
            print(f"    Max variance:           {field_stats.get('max_variance', 0):.4f}")
            print(f"    Avg duplication ratio:  {field_stats.get('avg_duplication_ratio', 0):.4f}")
    
    # Algorithm info
    if 'algorithm_info' in metrics:
        print("\n⚙️  Algorithm Configuration:")
        algo_info = metrics['algorithm_info']
        print(f"  Hash algorithm:                {algo_info.get('hash_algorithm_used', 'N/A')}")
        print(f"  Variance threshold:            {algo_info.get('variance_threshold', 0)}")
        print(f"  Large group threshold:         {algo_info.get('large_group_threshold', 0)}")
        print(f"  Large group variance thresh:   {algo_info.get('large_group_variance_threshold', 0)}")
    
    # Display result metrics
    print("\n" + "=" * 80)
    print("✨ SUMMARY")
    print("=" * 80)
    if result.metrics:
        for key, value in list(result.metrics.items())[:10]:
            if isinstance(value, (int, float)):
                if isinstance(value, float):
                    print(f"  {key:30s}: {value:.4f}")
                else:
                    print(f"  {key:30s}: {value}")
    
    print(f"\n💡 Groups with variance ≤ {operation.variance_threshold} can be aggregated!")
else:
    print("⚠️  No metrics file found in", task_dir / 'metrics')

## Step 6: Review Artifacts Location

**How to use:**
- Check all generated files in directory structure
- Navigate to task_dir for manual inspection
- Verify file counts and sizes

**What you'll see:**
- Directory structure with file counts per folder
- File names with sizes (KB) for up to 5 files per folder
- Full path to task directory for external access
- Total artifacts confirmation

**Output structure:**
```
examples/data_examples/simple_output/
├── metrics/          # Group analysis metrics JSON
├── visualizations/   # PNG charts
└── config/           # Operation config
```

In [ ]:
print("\n📂 OUTPUT DIRECTORY STRUCTURE:")
print("=" * 80)
print(f"\nTask directory: {task_dir.absolute()}\n")

for subdir in ['metrics', 'visualizations', 'config']:
    path = task_dir / subdir
    if path.exists():
        files = list(path.glob('*'))
        print(f"📁 {subdir}/ ({len(files)} files)")
        for f in files[:5]:
            size_kb = f.stat().st_size / 1024
            print(f"   • {f.name} ({size_kb:.1f} KB)")

print("\n✅ All artifacts saved successfully!")

## Step 7: Detailed Artifact Review

**How to use:**
- Review all generated artifacts in detail
- Automatically loads the NEWEST files from each category
- Displays visualizations inline for easy review

**What you'll see:**
1. **Metrics JSON**: Group variability metrics and statistics
2. **Visualizations**: Variance distribution histogram and field variability heatmap

**Note:** Only the most recent files are shown to avoid confusion from multiple runs

In [ ]:
print("\n📊 DETAILED ARTIFACT REVIEW")
print("=" * 80)

# 1. METRICS JSON (NEWEST)
print("\n1️⃣ METRICS (JSON):")
print("-" * 80)

metrics_dir = task_dir / 'metrics'
if not metrics_dir.exists():
    print(f"⚠️  Metrics directory not found: {metrics_dir}")
else:
    metrics_files = sorted(
        metrics_dir.glob("*.json"),
        key=lambda x: x.stat().st_mtime,
        reverse=True,
    )

    if not metrics_files:
        print("⚠️  No metrics files found")
    else:
        latest_metrics_file = metrics_files[0]
        mtime = datetime.fromtimestamp(latest_metrics_file.stat().st_mtime)

        print(f"📄 Loading LATEST: {latest_metrics_file.name}")
        print(f"🕒 Modified: {mtime:%Y-%m-%d %H:%M:%S}")
        print(f"📦 Size: {latest_metrics_file.stat().st_size / 1024:.1f} KB")
        print("-" * 80)

        try:
            with open(latest_metrics_file, "r", encoding="utf-8") as f:
                raw = json.load(f)

            metadata = raw.get("metadata", {})
            metrics = raw.get("metrics", {})

            # METADATA
            print("🧾 METADATA")
            print("-" * 80)
            print(f"Operation name : {metadata.get('name', 'N/A')}")
            print(f"Timestamp      : {metadata.get('timestamp', 'N/A')}")
            op = metadata.get("operation", {})
            print(f"Module         : {op.get('module', 'N/A')}")
            print(f"Class          : {op.get('class', 'N/A')}")
            print(f"Function       : {op.get('function', 'N/A')}")

            # OVERALL METRICS
            print("\n📊 OVERALL GROUP METRICS")
            print("-" * 80)
            print(f"Anchor field           : {metrics.get('field')}")
            print(f"Total records          : {metrics.get('total_records', 0):,}")
            print(f"Total groups           : {metrics.get('total_groups', 0):,}")
            print(f"Groups to aggregate    : {metrics.get('groups_to_aggregate', 0):,}")
            print(f"Average variance       : {metrics.get('avg_variance', 0):.4f}")
            print(f"Maximum variance       : {metrics.get('max_variance', 0):.4f}")

            agg_ratio = (
                metrics.get("groups_to_aggregate", 0)
                / max(metrics.get("total_groups", 1), 1)
            )
            print(f"Aggregation ratio      : {agg_ratio:.2%}")

            # VARIANCE DISTRIBUTION
            dist = metrics.get("threshold_metrics")
            if dist:
                print("\n📈 VARIANCE DISTRIBUTION")
                print("-" * 80)
                for k in [
                    "below_0.1",
                    "0.1_to_0.2",
                    "0.2_to_0.5",
                    "0.5_to_0.8",
                    "above_0.8",
                ]:
                    print(f"{k.replace('_', ' ').ljust(18)}: {dist.get(k, 0):>5}")

            # FIELD-LEVEL METRICS
            field_metrics = metrics.get("field_metrics", {})
            if field_metrics:
                print("\n🧩 FIELD CONTRIBUTION (Top fields)")
                print("-" * 80)
                for field, s in field_metrics.items():
                    print(f"\n• {field}")
                    print(f"   Avg variance        : {s.get('avg_variance', 0):.4f}")
                    print(f"   Max variance        : {s.get('max_variance', 0):.4f}")
                    print(f"   Avg duplication     : {s.get('avg_duplication_ratio', 0):.2f}")
                    print(f"   Unique values total : {s.get('unique_values_total', 0):,}")

            # SAMPLE GROUP METRICS (SAFE DISPLAY)
            group_metrics = metrics.get("group_metrics", {})
            if group_metrics:
                print("\n👥 SAMPLE GROUP METRICS (First 5)")
                print("-" * 80)

                for i, (gid, g) in enumerate(group_metrics.items()):
                    if i >= 5:
                        break

                    print(f"\nGroup ID: {gid}")
                    print(f"   Records             : {g.get('total_records')}")
                    print(f"   Weighted variance   : {g.get('weighted_variance', 0):.4f}")
                    print(f"   Max field variance  : {g.get('max_field_variance', 0):.4f}")
                    print(f"   Should aggregate    : {g.get('should_aggregate')}")

                    fv = g.get("field_variances", {})
                    if fv:
                        print("   Field variances:")
                        for f, v in fv.items():
                            print(f"      - {f}: {v:.4f}")

            # ALGORITHM PARAMETERS (AUDIT VALUE)
            algo = metrics.get("algorithm_info")
            if algo:
                print("\n⚙️  ALGORITHM PARAMETERS")
                print("-" * 80)
                for k, v in algo.items():
                    print(f"{k.replace('_', ' ').ljust(32)}: {v}")

            # FINAL INTERPRETATION
            print("\n🧠 INTERPRETATION")
            print("-" * 80)

            if agg_ratio > 0.5:
                print("⚠️  High aggregation pressure detected")
                print("   → Many groups show high internal variance")
                print("   → Direct identifiers may cause fragmentation")
            else:
                print("✅ Group structure relatively stable")

            print("   Suitable for:")
            print("   • k-anonymity preprocessing")
            print("   • Group generalization decisioning")
            print("   • Privacy-aware aggregation")

        except json.JSONDecodeError as e:
            print(f"❌ JSON decode error: {e}")
        except Exception as e:
            print(f"❌ Error reading metrics: {e}")

# 2. VISUALIZATIONS (NEWEST BATCH)
print("\n\n2️⃣ VISUALIZATIONS:")
print("-" * 80)

viz_dir = task_dir / 'visualizations'
if not viz_dir.exists():
    print(f"⚠️  Visualizations directory not found: {viz_dir}")
else:
    viz_files = list(viz_dir.glob('*.png'))
    
    if not viz_files:
        print("⚠️  No visualizations found")
    else:
        # Sort by modification time (newest first)
        viz_files.sort(key=lambda x: x.stat().st_mtime, reverse=True)
        
        # Get timestamp from latest file to identify the batch
        if viz_files:
            latest_time = viz_files[0].stat().st_mtime
            time_threshold = 10  # seconds
            latest_viz_batch = [
                f for f in viz_files 
                if abs(f.stat().st_mtime - latest_time) < time_threshold
            ]
            
            # Sort batch by name for consistent ordering
            latest_viz_batch.sort(key=lambda x: x.name)
            
            latest_mtime = datetime.fromtimestamp(latest_time)
            print(f"✓ Found {len(viz_files)} total visualization(s)")
            print(f"📄 Showing LATEST batch: {len(latest_viz_batch)} files")
            print(f"   Created: {latest_mtime.strftime('%Y-%m-%d %H:%M:%S')}\n")
            
            # Display each visualization from latest batch
            for i, viz_file in enumerate(latest_viz_batch, 1):
                # Extract readable name
                if 'variance_dist' in viz_file.name:
                    viz_name = "Variance Distribution Histogram"
                elif 'field_heatmap' in viz_file.name:
                    viz_name = "Field Variability Heatmap"
                else:
                    viz_name = viz_file.stem.replace('_', ' ').title()
                
                print(f"\n{i}. {viz_name}")
                print(f"   File: {viz_file.name}")
                print("-" * 60)
                try:
                    display(Image(filename=str(viz_file)))
                except Exception as e:
                    print(f"   ⚠️  Could not display: {e}")
        
        if len(viz_files) > len(latest_viz_batch):
            print(f"\n💡 Note: {len(viz_files) - len(latest_viz_batch)} older visualization(s) not shown")

print("\n\n" + "=" * 80)
print("✅ ARTIFACT REVIEW COMPLETE")
print("=" * 80)

## 🎯 Summary

**What you learned:**  
✅ Load data from examples/data_examples/  
✅ Setup environment with TaskReporter, DataSource, ProgressTracker  
✅ Configure group analysis with field weights  
✅ Execute using operation.execute()  
✅ Analyze results and review artifacts  

**Key takeaways:**
- Group variance measures how much fields vary within groups
- Lower variance means records are more similar (better for aggregation)
- Higher variance means records are more different (keep separate)
- Field weights allow prioritizing important fields in variance calculation
- Variance threshold determines which groups can be safely aggregated
- Large groups may use stricter thresholds to ensure quality
- Variance distribution shows overall data consistency patterns

## 📚 Next Steps

**Ready for advanced features?** Try:

📘 **`02_group_analyzer_advanced.ipynb`** includes:
- MinHash algorithm for fuzzy text matching
- Custom variance thresholds per field
- Cross-group relationship analysis
- Chunked processing for large datasets
- Advanced aggregation strategies

**Resources:**
- 📖 [PAMOLA Documentation](../../docs/)
- 🐙 [GitHub Repository](https://github.com/DGT-Network/PAMOLA)